# Experiment 9 — Cross-policy distinctiveness

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** A policy-specific model should predict its own policy better
than any other. Build a $6\times 6$ transfer-MAE matrix: fit on policy
$i$, test on policy $j$. Diagonal entries should dominate.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error

# Standardize everyone on the same feature map for comparability
COMMON_FEATURE = 'III'       # full lag-2 linear, covers everything

def fit_one_policy(policy):
    meta, X, y, cols = nu.build_design_matrix(turns, policy, COMMON_FEATURE)
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    y_logit = nu.logit_clip(y)
    reg = Ridge(alpha=1.0).fit(Xs, y_logit)
    return {'scaler': scaler, 'reg': reg, 'meta': meta, 'X': X, 'y': y}

fits = {p: fit_one_policy(p) for p in POLICIES}
mat  = pd.DataFrame(index=POLICIES, columns=POLICIES, dtype=float)
for src_p in POLICIES:
    src = fits[src_p]
    for tgt_p in POLICIES:
        tgt = fits[tgt_p]
        X_scaled = src['scaler'].transform(tgt['X'])
        yhat = nu.sigmoid(src['reg'].predict(X_scaled))
        mat.loc[src_p, tgt_p] = mean_absolute_error(tgt['y'], yhat)
print('Transfer MAE matrix  (rows = train policy, cols = test policy):')
print(mat.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(mat.to_numpy().astype(float), cmap='viridis_r', aspect='auto')
ax.set_xticks(range(len(POLICIES))); ax.set_xticklabels(POLICIES)
ax.set_yticks(range(len(POLICIES))); ax.set_yticklabels(POLICIES)
ax.set_xlabel('test policy'); ax.set_ylabel('train policy')
plt.colorbar(im, ax=ax, fraction=0.04, label='MAE (norm)')
for i in range(len(POLICIES)):
    for j in range(len(POLICIES)):
        v = float(mat.iat[i, j])
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8,
                color='white' if v > mat.values.astype(float).mean() else 'black')
ax.set_title('Cross-policy transfer MAE')
plt.tight_layout()
plt.savefig('figE9_transfer_matrix.png', dpi=140, bbox_inches='tight')
plt.show()

# Matched vs mismatched ratio
diag = np.diag(mat.to_numpy().astype(float))
off  = mat.to_numpy().astype(float).copy()
np.fill_diagonal(off, np.nan)
print(f'Mean diagonal (matched): {np.nanmean(diag):.4f}')
print(f'Mean off-diagonal (mismatched): {np.nanmean(off):.4f}')
print(f'Ratio mismatched/matched: {np.nanmean(off)/np.nanmean(diag):.2f}')